In [1]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

RIOT_PERSONAL_API_KEY = os.getenv("LOL_PERSONAL_API_KEY")
POSTGRES_LOCAL_URL: str = str(os.getenv("POSTGRES_LOCAL_URL"))

In [17]:
from typing import List, LiteralString
import psycopg
from psycopg import sql
import polars as pl


class PostgresConnection:
    def __init__(self, url: str):
        print("opening connection")
        self.connection: psycopg.Connection = psycopg.connect(url)

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc, tb):
        self.close()

    def close(self):
        print("closing connection")
        if self.connection and not self.connection.closed:
            self.connection.close()

    @classmethod
    def dict_to_tuple(cls, dictionary: dict) -> tuple:
        return tuple(
            cls.dict_to_tuple(value) if isinstance(value, dict) else value
            for value in dictionary.values()
        )

    def upsert_dataframe_dto_safe(
            self,
            dataframe: pl.DataFrame,
            table_name: str,
            conflict_columns: List[str]
    ):
        if dataframe.is_empty():
            return 1

        columns = dataframe.columns
        update_columns = [col for col in columns if col not in conflict_columns]

        if len(update_columns) == 0:
            return 1

        sql_operation = sql.SQL(
        """
          INSERT INTO {table_name} ({column_list_str})
          VALUES ({bind_arguments_str})
          ON CONFLICT ({conflict_clause_str})
          DO UPDATE SET {update_clause_str}
        """
        ).format(
            table_name=sql.Identifier(table_name),
            column_list_str = sql.SQL(", ").join(map(sql.Identifier, columns)),
            bind_arguments_str = sql.SQL(", ").join(sql.Placeholder() * len(columns)),
            conflict_clause_str = sql.SQL(", ").join(map(sql.Identifier, conflict_columns)),
            update_clause_str = sql.SQL(", ").join(
                sql.SQL("{col} = EXCLUDED.{col}").format(col=sql.Identifier(col))
                for col in update_columns
            ),
        )

        with self.connection.transaction():
            with self.connection.cursor() as cursor:
                cursor.executemany(
                    sql_operation,
                    dataframe.with_columns(
                        pl.col(pl.Struct).map_elements(
                            self.dict_to_tuple,
                            return_dtype=pl.Object
                        )
                    )
                    .rows()
                )

        return 0


In [2]:
import json
from datetime import datetime, date
from dataclasses import dataclass
import polars as pl


@dataclass
class ChallengePointDto:
    level: str
    current: int
    max: int
    percentile: float

In [6]:
test_df = (pl.DataFrame(
    {
        "puuid": "x1jqYLksFHSjj-V_dTeZrzR9fFEVDtcRFuZ0u9novIzvQW-QF-dE9YyTxsi08KU5rAUNccN9F2wLXQ",
        "snapshot_date": date.today(),
        "summoner_level": 25,
        "title": "caca",
        "profile_icon_id": 1,
        "crest_border": "ceva",
        "banner_accent": "ceva",
        "prestige_crest_border_level": 1,
        "total_points": ChallengePointDto(
            level="test",
            current=1,
            max=1,
            percentile=50.0,
        )
    }
)
)

In [4]:
from utils.postgres.postgres_connection import PostgresConnection
connection_wrapper = PostgresConnection(POSTGRES_LOCAL_URL)

opening connection


In [7]:
connection_wrapper.upsert_dataframe_dto_safe(test_df, "player_snapshots", ["puuid", "snapshot_date"])

0

closing connection
